# Benchmark query generation (decision 0021)

Sarvam-M writes a backstory and a query for each of the 200 benchmark slots
(`data/queries/benchmark_cells_v1.jsonl`, rank 0). It is the model the author's blind ratings chose
(`results/queries/generator_trial_v1/summary_v1.md`), loaded at the same revision, with the same prompts, 4-bit
loading and greedy decoding as the trial, so the trial's ratings describe these outputs.

Reserves (ranks 1 and 2) are generated only for slots the author rejects at curation: list them in
`data/queries/reserve_requests_v1.json` (written by `src.queries.curation`), commit, and run this notebook again.

**Before running**
- Runtime: GPU (Colab T4 or Kaggle). One model, about 47 GB of disk while it downloads.
- No token is needed; a Hugging Face token saved as the secret `HF_TOKEN` (Colab) only speeds up the download.

**Output**: `query_generation_v1/outputs.jsonl`, one line per cell, with the model and revision, prompts, raw and
cleaned outputs, generation settings, library versions, GPU and date. Download it and place it at
`results/queries/generation_v1/outputs.jsonl` in the repository. The run can be restarted: cells already done are
skipped.

**If a run was cut off:** it stops cleanly at 11 hours. Start a new version with Add Input, this notebook's
earlier output; it continues from the saved `outputs.jsonl`. Cells are generated in batches of 8 with left padding;
each output records its batch size.

In [ ]:
REPO = "https://github.com/azzs11/heritagerec.git"
BRANCH = "main"
MODEL = "sarvamai/sarvam-m"  # Apache-2.0
# The revision the author rated in the trial; pinned so the benchmark is written by exactly that model.
REVISION = "01534a53c46f2788e392dbb3d994e0fa8f04d3fd"
# Same settings as the trial (greedy for reproducibility, thinking off).
GENERATION = {"do_sample": False, "max_new_tokens_backstory": 200, "max_new_tokens_query": 64}
OUT_DIR = "query_generation_v1"
# Cells generated together (left-padded, greedy): far faster than one at a time on the same GPUs. Recorded per output.
BATCH = 8
# Kaggle stops a version at 12 hours and may then keep nothing; stop cleanly before that, so the outputs are saved,
# and continue in a new version from them (attach this version's output as input).
STOP_AFTER_HOURS = 11.0

In [ ]:
!git clone --depth 1 -b $BRANCH $REPO heritagerec
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

In [ ]:
# Optional: a token only raises download rate limits; none of the models is gated.
try:
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Hugging Face token used:", type(e).__name__)

In [ ]:
import glob
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import bitsandbytes
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

sys.path.insert(0, "heritagerec")
from src.queries.prompts import PROMPT_VERSION, backstory_prompt, query_prompt

all_cells = [json.loads(line) for line in open("heritagerec/data/queries/benchmark_cells_v1.jsonl", encoding="utf-8")]
requests_path = Path("heritagerec/data/queries/reserve_requests_v1.json")
# A request names a slot and the reserve rank to use for it, e.g. {"q017": 1}.
reserve_requests = json.loads(requests_path.read_text()) if requests_path.exists() else {}
cells = [c for c in all_cells if c["rank"] == 0 or reserve_requests.get(c["slot"]) == c["rank"]]
repo_commit = subprocess.run(
    ["git", "-C", "heritagerec", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()
out_path = Path(OUT_DIR) / "outputs.jsonl"
out_path.parent.mkdir(exist_ok=True)
# Continue from an earlier version's outputs when attached as input (Kaggle: Add Input, this notebook's output).
if not out_path.exists():
    earlier = sorted(glob.glob("/kaggle/input/**/query_generation_v1/outputs.jsonl", recursive=True))
    if earlier:
        shutil.copy(earlier[-1], out_path)
        print("continuing from", earlier[-1])
done = {r["cell_id"] for r in map(json.loads, out_path.open(encoding="utf-8"))} if out_path.exists() else set()
environment = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "repo_commit": repo_commit,
}
print(
    len(cells), "cells to write;", len(reserve_requests), "reserve requests;", len(done), "already saved;", environment
)

In [ ]:
import time


def strip_thinking(text: str) -> str:
    # Thinking is off, but drop an empty or stray think block if a template still emits one.
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text.strip()


def clean_query(text: str) -> str:
    # Keep the first non-empty line and drop wrapping quotes; the raw output is saved too.
    text = strip_thinking(text)
    line = next((x for x in text.splitlines() if x.strip()), "").strip()
    return line.strip("\"'“”‘’`").strip()


def generate_batch(prompts: list[str], max_new_tokens: int) -> list[str]:
    # The same single user turn as the trial, for several cells at once. The chat template already holds the
    # begin-of-text token, so the tokenizer adds none; padding goes on the left so every answer starts at the end.
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], add_generation_prompt=True, enable_thinking=False, tokenize=False
        )
        for p in prompts
    ]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=GENERATION["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
        )
    start = enc["input_ids"].shape[1]
    return [tokenizer.decode(o[start:], skip_special_tokens=True).strip() for o in out]


todo = [c for c in cells if c["cell_id"] not in done]
compute = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, quantization_config=quant, device_map="auto")
print(len(todo), "cells to generate in batches of", BATCH, flush=True)

began = time.time()
for i in range(0, len(todo), BATCH):
    if time.time() - began > STOP_AFTER_HOURS * 3600:
        print(
            f"stopping cleanly after {STOP_AFTER_HOURS} h; {len(todo) - i} cells left for the next version", flush=True
        )
        break
    batch = todo[i : i + BATCH]
    p1 = [backstory_prompt(c) for c in batch]
    backstories_raw = generate_batch(p1, GENERATION["max_new_tokens_backstory"])
    backstories = [strip_thinking(b) for b in backstories_raw]
    p2 = [query_prompt(c, b) for c, b in zip(batch, backstories)]
    queries_raw = generate_batch(p2, GENERATION["max_new_tokens_query"])
    with out_path.open("a", encoding="utf-8") as f:
        for cell, a, b_raw, b, q, q_raw in zip(batch, p1, backstories_raw, backstories, p2, queries_raw):
            record = {
                "cell_id": cell["cell_id"],
                "cell": cell,
                "model": MODEL,
                "revision": REVISION,
                "quantization": f"4-bit nf4, compute {str(compute).replace('torch.', '')}",
                "prompt_version": PROMPT_VERSION,
                "backstory_prompt": a,
                "backstory_raw": b_raw,
                "backstory": b,
                "query_prompt": q,
                "query_raw": q_raw,
                "query": clean_query(q_raw),
                "generation": GENERATION | {"batch_size": BATCH, "padding": "left"},
                "environment": environment,
                "date": datetime.now(timezone.utc).isoformat(),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            done.add(cell["cell_id"])
            print(cell["cell_id"], cell["language"], cell["query_type"], "|", record["query"], flush=True)
    finished = i + len(batch)
    rate = (time.time() - began) / finished
    print(f"-- {finished} of {len(todo)} done; about {rate * (len(todo) - finished) / 60:.0f} min left", flush=True)

Check the printed queries for empty or garbled output; do not rerun with different settings. Problems are handled
at curation (edit or reject, with the reason logged).

In [ ]:
print(sum(1 for _ in out_path.open(encoding="utf-8")), "outputs")
try:
    from google.colab import files

    files.download(str(out_path))
except ImportError:
    # On Kaggle the file is in the Output panel (right side) under query_generation_v1/.
    print("Download", out_path.resolve(), "from the notebook's output files")